In [ ]:
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv pillow google-generativeai

import os
import base64
import io
import asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

app = FastAPI()

# CORS 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 환경 변수에서 모델 모드 확인
useModel = os.getenv("USE_MODEL", "GEMINI")

def convertImageToBase64(imageBytes):
    """이미지 바이트를 base64 문자열로 변환"""
    return base64.b64encode(imageBytes).decode("utf-8")

@app.get("/")
async def root():
    """서버 상태 확인"""
    return {"message": "서버 온라인", "model": useModel}

@app.post("/analyze")
async def analyzeImage(
    image: UploadFile = File(...),
    question: str = Form(default="이 이미지를 분석해줘")
):
    """이미지와 질문을 받아 AI 모델로 분석"""
    try:
        # 이미지 읽기
        imageBytes = await image.read()
        imageBase64 = convertImageToBase64(imageBytes)

        responseText = ""

        if useModel == "OLLAMA":
            # OLLAMA 모델로 분석
            import ollama
            response = ollama.chat(
                model="gemma4:e2b",
                messages=[
                    {
                        "role": "user",
                        "content": question,
                        "images": [imageBase64]
                    }
                ]
            )
            responseText = response["message"]["content"]

        else:
            # GEMINI 모델로 분석
            import google.generativeai as genai
            from PIL import Image

            # API 키 설정
            geminiApiKey = os.getenv("GEMINI_API_KEY")
            genai.configure(api_key=geminiApiKey)

            # 이미지 변환
            imageData = Image.open(io.BytesIO(imageBytes))

            # Gemini 모델 호출
            model = genai.GenerativeModel("gemini-2.0-flash")
            response = model.generate_content([question, imageData])
            responseText = response.text

        return {
            "success": True,
            "model": useModel,
            "question": question,
            "answer": responseText
        }

    except Exception as e:
        return {"success": False, "message": f"에러 발생: {str(e)}"}

# Jupyter에서 실행
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [12824]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52368 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:52368 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:63194 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:59868 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:55897 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:55897 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:52207 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:65279 - "POST /analyze HTTP/1.1" 200 OK
INFO:     127.0.0.1:59998 - "POST /analyze HTTP/1.1" 200 OK
